# RIM (Residual Income Model) Valuation
## Sales-driven DuPont ROE → Ohlson (1995) Residual Income Valuation

### Model Flow
```
①  DB (us_revenue_forecast_data)  →  Actual + 8Q Forecast Sales
②  FMP API  →  IS / BS quarterly data
③  DuPont decomposition (Sales-driven)
      NPM   = Net Income / Sales     [OLS or median]
      AT    = Sales / Total Assets   [OLS or median]
      FL    = Total Assets / Equity  [OLS or median]
      ROE   = NPM × AT × FL          [Phase 1: 2yr forecast]
④  Re  from DB (us_rim_spread_data.Re_smooth)
⑤  RI spread decay  →  AR(1) Ohlson omega  (EVA/moat-based)
      Phase 2 (5~15yr): RI(t) = omega × RI(t-1)
      Phase 3: TV = RI_last / (Re - g)
⑥  Intrinsic Value = Book Value + PV(RI Phase1) + PV(RI Phase2) + PV(TV)
```

### Academic References
- **Ohlson (1995)** *JAR* — Residual Income / Linear Information Model
- **Feltham & Ohlson (1995)** *JAR* — ω (persistence) coefficient
- **Fama & French (1995)** *JF* — Mean reversion of profitability (AR(1))
- **Mauboussin & Johnson (1997)** *FAJ* — Competitive Advantage Period
- **Damodaran (2002)** — 3-Stage structure & phase period guidance


## Cell 1 · Path Setup

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root auto-detected : {root}")
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root fallback : {candidate}")
            return candidate
    raise EnvironmentError("DATA folder not found. Update _CANDIDATE_ROOTS.")

_ROOT = _setup_path()
print(f"[OK] Project root : {_ROOT}")


## Cell 2 · Imports & Constants

> **Edit here only**: `TICKER_START / TICKER_END / SKIP_DONE`


In [ ]:
import gc, math, time, traceback, warnings
import requests
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as US_TICKER_LIST

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ── FMP ───────────────────────────────────────────────────────────────────────
FMP_API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE    = "https://financialmodelingprep.com/api/v3"
FMP_LIMIT   = 40
FMP_SLEEP   = 0.35

# ── DB Tables ─────────────────────────────────────────────────────────────────
TABLE_SALES   = "us_revenue_forecast_data"   # sales forecast
TABLE_RE      = "us_rim_spread_data"          # Re_smooth (already computed)
TABLE_RESULT  = "us_rim_valuation"            # RIM output
DEFAULT_PORT  = 3307

# ── Model Parameters ──────────────────────────────────────────────────────────
FORECAST_HORIZON = 8       # quarters for Phase 1 (2 years)
MIN_HISTORY      = 12      # minimum quarters for DuPont OLS
OLS_MIN_R2       = 0.20    # relaxed vs FCFF (ratio regression is noisier)
OLS_MIN_SAMPLES  = 12
WINSORIZE_LIMITS = (0.05, 0.95)
GDP_GROWTH       = 0.04    # terminal growth cap
RETENTION_FLOOR  = 0.20    # minimum earnings retention ratio

# ── Batch Range ───────────────────────────────────────────────────────────────
TICKER_START = 0
TICKER_END   = 10          # <- adjust
SKIP_DONE    = False

CHECKPOINT_DIR = "_rim_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

db_info = get_db_info()
engine  = get_engine(db_info)

print(f"[OK] Imports complete  |  tickers: {len(US_TICKER_LIST):,}")
print(f"[Config] FORECAST_HORIZON={FORECAST_HORIZON}Q  GDP_GROWTH={GDP_GROWTH}")


## Cell 3 · DB Connection & Table Init

In [ ]:
def get_conn():
    return pymysql.connect(
        host       = db_info["host"],
        port       = int(db_info.get("port", DEFAULT_PORT)),
        user       = db_info["user"],
        password   = db_info["password"],
        db         = db_info.get("database", "investar"),
        charset    = "utf8mb4",
        autocommit = False,
        cursorclass= pymysql.cursors.DictCursor,
    )

CREATE_SQL = """
CREATE TABLE IF NOT EXISTS `us_rim_valuation` (
  `id`              BIGINT      NOT NULL AUTO_INCREMENT,
  `date`            DATE        NOT NULL COMMENT 'run date',
  `ticker`          VARCHAR(20) NOT NULL,
  `year_label`      VARCHAR(10)          COMMENT 'e.g. 2026 or Ph2-Y3',
  `phase`           VARCHAR(10)          COMMENT 'ph1 / ph2 / tv',
  `sales_forecast`  DOUBLE               COMMENT 'annual sales forecast',
  `npm_forecast`    DOUBLE               COMMENT 'net profit margin',
  `asset_turnover`  DOUBLE,
  `fin_leverage`    DOUBLE,
  `roe_forecast`    DOUBLE,
  `re`              DOUBLE               COMMENT 'cost of equity',
  `ri_spread`       DOUBLE               COMMENT 'ROE - Re',
  `bv_start`        DOUBLE               COMMENT 'book value (start of year)',
  `ri`              DOUBLE               COMMENT 'residual income',
  `pv_ri`           DOUBLE               COMMENT 'PV of RI',
  `moat_label`      VARCHAR(30),
  `rho`             DOUBLE               COMMENT 'AR(1) persistence coef',
  `n_phase2`        INT,
  `g_terminal`      DOUBLE,
  `pv_all_ri`       DOUBLE,
  `terminal_value`  DOUBLE,
  `intrinsic_value` DOUBLE,
  `current_price`   DOUBLE,
  `upside_pct`      DOUBLE,
  `target_price`    DOUBLE,
  `created_at`      DATETIME    DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`, `year_label`),
  INDEX idx_ticker (`ticker`),
  INDEX idx_date   (`date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    log("DB", f"Table ready: {TABLE_RESULT}")
finally:
    conn.close()

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"Connected  host={db_info.get('host')}  port={db_info.get('port')}")
except Exception as e:
    log("DB", f"Connection failed: {e}")


## Cell 4 · RIMModel Class

### Structure
| Method | Role |
|--------|------|
| `load_sales()` | DB → actual + forecast Sales |
| `load_financials()` | FMP → IS / BS quarterly |
| `load_re()` | DB → Re_smooth (cost of equity) |
| `estimate_retention()` | earnings retention ratio |
| `estimate_dupont_coefs()` | NPM / AT / FL via OLS or median |
| `forecast_roe_phase1()` | DuPont ROE for Phase 1 (2yr) |
| `compute_eva_spread()` | ROIC - WACC for moat classification |
| `moat_to_omega_and_years()` | EVA → (ω, Phase2 years, label) |
| `compute_ri_path()` | full RI time-series (Ph1 + Ph2) |
| `compute_valuation()` | BV + PV(RI) + PV(TV) |
| `save_to_db()` | upsert to us_rim_valuation |
| `plot()` | historical + forecast RI / ROE visualisation |


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  RIMModel — Ohlson (1995) Residual Income Valuation
#  Sales-driven DuPont ROE  +  AR(1) RI spread decay
# ═══════════════════════════════════════════════════════════════

class RIMModel:
    """
    Residual Income Model (Ohlson 1995) with Sales-driven DuPont forecasting.

    Valuation:
        P = BV_0 + sum( RI_t / (1+Re)^t ) + PV(TV)
        RI_t = (ROE_t - Re) * BV_{t-1}
        BV_t = BV_{t-1} * (1 + ROE_t * retention_t)

    Phase 1 (2yr): Sales forecast → DuPont → ROE
    Phase 2 (5~15yr): AR(1) RI decay  RI(t) = omega * RI(t-1)
                      omega determined by EVA moat (Ohlson 1995, Fama-French 1995)
    Phase 3: TV = RI_last / (Re - g)  [if RI_last > 0]

    References:
        Ohlson (1995) JAR, Feltham & Ohlson (1995) JAR,
        Fama & French (1995) JF, Mauboussin & Johnson (1997) FAJ,
        Damodaran (2002) Investment Valuation
    """

    def __init__(
        self,
        ticker: str,
        engine,
        fmp_api_key: str = FMP_API_KEY,
        forecast_horizon: int = FORECAST_HORIZON,
        min_history: int = MIN_HISTORY,
        gdp_growth: float = GDP_GROWTH,
        verbose: bool = False,
    ):
        self.ticker          = ticker.upper()
        self.engine          = engine
        self.api_key         = fmp_api_key
        self.horizon         = forecast_horizon
        self.min_history     = min_history
        self.gdp_growth      = gdp_growth
        self.verbose         = verbose

        self._sales_actual:   Optional[pd.Series] = None
        self._sales_forecast: Optional[pd.Series] = None
        self._inc:            Optional[pd.DataFrame] = None
        self._bs:             Optional[pd.DataFrame] = None
        self._re:             Optional[float] = None
        self._eva_cache:      Optional[dict] = None
        self.result_df:       Optional[pd.DataFrame] = None
        self.valuation:       Optional[Dict] = None

    # ─────────────────────────────────────────────────────────
    # 1. Data Loading
    # ─────────────────────────────────────────────────────────

    def load_sales(self) -> "RIMModel":
        """
        Load Sales from DB (us_revenue_forecast_data).
        Identical pattern to DCFModel.load_sales() — cursor.execute/fetchall only.
        """
        conn_local = get_conn()
        try:
            with conn_local.cursor() as cur:
                cur.execute(
                    f"SELECT date, data_type, model, value "
                    f"FROM `{TABLE_SALES}` "
                    f"WHERE ticker=%s AND item='sale' AND data_type='actual' "
                    f"ORDER BY date",
                    (self.ticker,)
                )
                actual_rows = cur.fetchall()

            with conn_local.cursor() as cur:
                cur.execute(
                    f"SELECT MAX(forecast_date) AS max_fd "
                    f"FROM `{TABLE_SALES}` "
                    f"WHERE ticker=%s AND item='sale' AND data_type='forecast'",
                    (self.ticker,)
                )
                row = cur.fetchone()
                max_fd = row["max_fd"] if row else None

            fc_rows = []
            if max_fd:
                with conn_local.cursor() as cur:
                    cur.execute(
                        f"SELECT date, data_type, model, value "
                        f"FROM `{TABLE_SALES}` "
                        f"WHERE ticker=%s AND item='sale' "
                        f"  AND data_type='forecast' AND forecast_date=%s "
                        f"ORDER BY model, date",
                        (self.ticker, max_fd)
                    )
                    fc_rows = cur.fetchall()
        finally:
            conn_local.close()

        if not actual_rows:
            raise ValueError(f"[{self.ticker}] No actual Sales — run revenue forecast notebook first")

        act_df = pd.DataFrame(actual_rows)
        act_df["date"]  = pd.to_datetime(act_df["date"])
        act_df["value"] = pd.to_numeric(act_df["value"], errors="coerce")
        actual = (act_df.sort_values("date")
                        .drop_duplicates("date", keep="last")
                        .set_index("date")["value"])

        if not fc_rows:
            raise ValueError(f"[{self.ticker}] No forecast Sales (forecast_date={max_fd})")

        fc_df = pd.DataFrame(fc_rows)
        fc_df["date"]  = pd.to_datetime(fc_df["date"])
        fc_df["value"] = pd.to_numeric(fc_df["value"], errors="coerce")

        priority  = ["Ensemble", "SARIMA", "ETS", "Theta"]
        available = fc_df["model"].unique().tolist()
        ordered   = [m for m in priority if m in available] +                     [m for m in available if m not in priority]

        forecast = pd.Series(dtype=float)
        for model_name in ordered:
            fc = (fc_df[fc_df["model"] == model_name]
                  .sort_values("date")
                  .drop_duplicates("date", keep="last")
                  .set_index("date")["value"].dropna())
            if len(fc) >= self.horizon:
                forecast = fc.iloc[:self.horizon]
                break
            elif len(fc) > 0 and forecast.empty:
                forecast = fc

        if forecast.empty:
            raise ValueError(f"[{self.ticker}] Forecast Sales empty")

        if len(forecast) < self.horizon:
            extra_idx = pd.date_range(
                start=forecast.index[-1] + pd.offsets.QuarterEnd(1),
                periods=self.horizon - len(forecast), freq="QE"
            )
            forecast = pd.concat([
                forecast,
                pd.Series([float(forecast.iloc[-1])] * len(extra_idx), index=extra_idx)
            ])

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:self.horizon]
        if self.verbose:
            log(self.ticker, f"Sales loaded  actual={len(actual)}Q  forecast={len(self._sales_forecast)}Q")
        return self

    def _fmp_get(self, endpoint: str, period: str = "quarter", limit: int = FMP_LIMIT) -> pd.DataFrame:
        url = f"{FMP_BASE}/{endpoint}/{self.ticker}"
        for attempt in range(3):
            try:
                r = requests.get(url, params={"period": period, "limit": limit,
                                              "apikey": self.api_key}, timeout=20)
                if r.status_code == 429:
                    time.sleep(1.5 + attempt); continue
                r.raise_for_status()
                data = r.json()
                if isinstance(data, dict) and "Error Message" in data:
                    return pd.DataFrame()
                return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()
            except Exception as e:
                if attempt == 2:
                    if self.verbose: log(self.ticker, f"FMP {endpoint} failed: {e}")
                    return pd.DataFrame()
                time.sleep(FMP_SLEEP + attempt * 0.5)
        return pd.DataFrame()

    def _parse_date_col(self, df: pd.DataFrame) -> pd.DataFrame:
        """acceptedDate → look-ahead-bias-free quarter-end date (same as DCFModel)"""
        df = df.copy()
        df["report_date"] = pd.to_datetime(
            df.get("acceptedDate", df.get("fillingDate", pd.NaT)), errors="coerce"
        )
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        def _qend(row):
            if pd.notna(row.get("report_date")):
                return (row["report_date"] - pd.Timedelta(days=45)).to_period("Q").to_timestamp("Q")
            return row["date"].to_period("Q").to_timestamp("Q")
        df["date"] = df.apply(_qend, axis=1)
        return df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

    def load_financials(self) -> "RIMModel":
        """FMP API → IS + BS (quarterly)"""
        time.sleep(FMP_SLEEP)
        inc = self._fmp_get("income-statement")
        time.sleep(FMP_SLEEP)
        bs  = self._fmp_get("balance-sheet-statement")
        if inc.empty or bs.empty:
            raise ValueError(f"[{self.ticker}] FMP IS/BS load failed")
        self._inc = self._parse_date_col(inc)
        self._bs  = self._parse_date_col(bs)
        if self.verbose:
            log(self.ticker, f"IS={len(self._inc)}Q / BS={len(self._bs)}Q")
        return self

    def load_re(self) -> "RIMModel":
        """Load Re_smooth from DB (us_rim_spread_data)"""
        sql = text(f"""
            SELECT value FROM `{TABLE_RE}`
            WHERE ticker=:t AND indicator='Re_smooth'
            ORDER BY date DESC LIMIT 1
        """)
        try:
            with self.engine.connect() as conn:
                row = conn.execute(sql, {"t": self.ticker}).fetchone()
            if row:
                self._re = float(row[0])
                if self.verbose: log(self.ticker, f"Re_smooth={self._re:.4f}")
                return self
        except Exception:
            pass
        # CAPM fallback: rf=4.5% + beta=1.0 * ERP=5.5%
        self._re = 0.045 + 1.0 * 0.055
        if self.verbose: log(self.ticker, f"Re_smooth not found → CAPM fallback Re={self._re:.4f}")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. DuPont Component Estimation
    # ─────────────────────────────────────────────────────────

    @staticmethod
    def _winsorize(s: pd.Series, limits=WINSORIZE_LIMITS) -> pd.Series:
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x: pd.Series, y: pd.Series) -> Tuple[float, float]:
        """Simple OLS y = slope*x. Returns (slope, R²)"""
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return slope, r ** 2

    def estimate_retention(self) -> float:
        """
        Earnings retention ratio = 1 - payout_ratio
        = (NetIncome - Dividends) / NetIncome
        Fallback: 0.60 (typical large-cap)
        """
        inc = self._inc
        bs  = self._bs
        needed_i = ["netIncome"]
        needed_b = ["dividendsPaid", "commonStockRepurchased"]

        if "netIncome" not in inc.columns:
            return 0.60

        ni = pd.to_numeric(inc["netIncome"], errors="coerce")
        ni_pos = ni[ni > 0]
        if ni_pos.empty:
            return 0.60

        # try dividends from BS or IS
        div = pd.Series(0.0, index=inc.index)
        if "dividendsPaid" in bs.columns:
            merged = inc[["date"]].merge(bs[["date","dividendsPaid"]], on="date", how="left")
            div = pd.to_numeric(merged["dividendsPaid"], errors="coerce").fillna(0).abs()

        payout = (div / ni.abs()).replace([np.inf, -np.inf], np.nan).dropna()
        payout = payout[payout.between(0, 1)]
        if payout.empty:
            return 0.60

        retention = float(1 - self._winsorize(payout).median())
        retention = float(np.clip(retention, RETENTION_FLOOR, 0.99))
        if self.verbose: log(self.ticker, f"Retention ratio={retention:.3f}")
        return retention

    def estimate_dupont_coefs(self) -> Dict[str, Any]:
        """
        Estimate DuPont components vs Sales (quarterly):
          NPM (net profit margin) = NetIncome / Sales
          AT  (asset turnover)    = Sales / TotalAssets  [annual proxy]
          FL  (financial leverage)= TotalAssets / TotalEquity

        Method: OLS slope (R²≥0.20) → median ratio fallback
        Returns dict with 'npm_slope','at_median','fl_median','method'
        """
        inc = self._inc.copy()
        bs  = self._bs.copy()

        for col in ["revenue","netIncome"]:
            if col not in inc.columns:
                inc[col] = np.nan
            inc[col] = pd.to_numeric(inc[col], errors="coerce")

        for col in ["totalAssets","totalStockholdersEquity"]:
            if col not in bs.columns:
                bs[col] = np.nan
            bs[col] = pd.to_numeric(bs[col], errors="coerce")

        # ── NPM: netIncome / revenue ──────────────────────────────
        df_npm = inc[["date","revenue","netIncome"]].dropna()
        df_npm = df_npm[df_npm["revenue"] > 0]
        npm_series = (df_npm["netIncome"] / df_npm["revenue"])
        npm_series = self._winsorize(npm_series.dropna())
        npm_median = float(npm_series.median()) if not npm_series.empty else 0.05

        # OLS slope (netIncome ~ revenue)
        if len(df_npm) >= OLS_MIN_SAMPLES:
            slope_npm, r2_npm = self._ols_ratio(df_npm["revenue"], df_npm["netIncome"])
            use_ols = (not np.isnan(slope_npm) and r2_npm >= OLS_MIN_R2)
        else:
            use_ols = False

        npm_coef   = float(slope_npm) if use_ols else npm_median
        npm_method = "ols" if use_ols else "median"

        # ── AT: sales / totalAssets (annual, TTM sales / year-end assets) ──
        merged_at = inc[["date","revenue"]].merge(bs[["date","totalAssets"]], on="date", how="inner")
        merged_at = merged_at.dropna()
        merged_at = merged_at[merged_at["totalAssets"] > 0]
        if not merged_at.empty:
            at_series  = self._winsorize((merged_at["revenue"] / merged_at["totalAssets"]).dropna())
            at_median  = float(at_series.median())
        else:
            at_median = 0.70  # broad fallback

        # ── FL: totalAssets / totalStockholdersEquity ──────────────
        bs_clean = bs[["date","totalAssets","totalStockholdersEquity"]].dropna()
        bs_clean = bs_clean[(bs_clean["totalAssets"] > 0) & (bs_clean["totalStockholdersEquity"] > 0)]
        if not bs_clean.empty:
            fl_series = self._winsorize((bs_clean["totalAssets"] / bs_clean["totalStockholdersEquity"]).dropna())
            fl_median = float(fl_series.median())
            fl_median = float(np.clip(fl_median, 1.0, 20.0))
        else:
            fl_median = 2.5   # fallback

        if self.verbose:
            log(self.ticker,
                f"DuPont coefs  NPM={npm_coef:.4f}({npm_method})  "
                f"AT={at_median:.3f}(median)  FL={fl_median:.2f}(median)")

        return {
            "npm_coef":   npm_coef,
            "npm_method": npm_method,
            "at_median":  at_median,
            "fl_median":  fl_median,
        }

    def forecast_roe_phase1(self, coefs: dict) -> pd.DataFrame:
        """
        Phase 1 ROE forecast (quarterly → annual aggregation):
        1. Quarterly sales forecast → Annual sales (4Q sum)
        2. NetIncome = npm_coef × Sales
        3. AT_est × FL_est → ROE = NPM × AT × FL
        4. Returns DataFrame with columns:
           [year, sales_annual, net_income, npm, at, fl, roe, bv_start, ri, ri_spread]
        """
        fc_q  = self._sales_forecast  # 8 quarters
        re    = self._re

        # Annual aggregation
        annual_sales = []
        for yr in range(0, len(fc_q), 4):
            chunk = fc_q.iloc[yr: yr + 4]
            annual_sales.append({
                "year":  fc_q.index[yr].year,
                "sales": float(chunk.sum()),
            })

        # Latest book value
        bs = self._bs.copy()
        if "totalStockholdersEquity" not in bs.columns:
            raise ValueError(f"[{self.ticker}] totalStockholdersEquity missing")
        bs["eq"] = pd.to_numeric(bs["totalStockholdersEquity"], errors="coerce")
        bv0 = float(bs.dropna(subset=["eq"])["eq"].iloc[-1])
        if bv0 <= 0:
            raise ValueError(f"[{self.ticker}] Book Value non-positive ({bv0:.1f})")

        retention = self.estimate_retention()

        rows = []
        bv_start = bv0
        for yr_info in annual_sales:
            sales  = yr_info["sales"]
            ni     = coefs["npm_coef"] * sales
            npm    = ni / sales if sales > 0 else 0.0
            # AT: revenue / assets  — assets estimated from equity * FL
            at_est = coefs["at_median"]
            fl_est = coefs["fl_median"]
            roe    = float(np.clip(npm * at_est * fl_est, -0.60, 1.00))

            ri_spread = roe - re
            ri        = ri_spread * bv_start

            rows.append({
                "year":         yr_info["year"],
                "phase":        "ph1",
                "sales_annual": sales,
                "net_income":   ni,
                "npm":          npm,
                "at":           at_est,
                "fl":           fl_est,
                "roe":          roe,
                "re":           re,
                "ri_spread":    ri_spread,
                "bv_start":     bv_start,
                "ri":           ri,
            })
            # update BV: BV(t) = BV(t-1) * (1 + ROE * retention)
            bv_start = bv_start * (1 + max(roe, 0) * retention)

        return pd.DataFrame(rows), bv_start   # bv_start after Ph1

    # ─────────────────────────────────────────────────────────
    # 3. EVA Spread & Moat Classification
    # ─────────────────────────────────────────────────────────

    def compute_wacc_simple(self) -> float:
        """
        Simplified WACC for EVA calculation.
        Re from DB + Rd from IS/BS. Same as DCFModel.compute_wacc().
        """
        re  = self._re or 0.10
        tax = self._estimate_tax_rate()

        bs  = self._bs
        inc = self._inc
        rd  = 0.05

        if "totalDebt" in bs.columns and "interestExpense" in inc.columns:
            latest     = bs.sort_values("date").iloc[-1]
            total_debt = pd.to_numeric(latest.get("totalDebt", 0), errors="coerce") or 0
            ie         = pd.to_numeric(inc["interestExpense"], errors="coerce").abs()
            td_avg     = pd.to_numeric(bs["totalDebt"], errors="coerce")
            td_avg     = (td_avg + td_avg.shift(1)) / 2
            valid      = (td_avg > 0) & ie.notna()
            if valid.sum() >= 2:
                rd = float((ie[valid] / td_avg[valid]).clip(0, 0.15).median())

            mkt_cap = 0.0
            try:
                r = requests.get(f"{FMP_BASE}/market-capitalization/{self.ticker}",
                                 params={"apikey": self.api_key}, timeout=10)
                d = r.json()
                if isinstance(d, list) and d:
                    mkt_cap = float(d[0].get("marketCap", 0) or 0)
            except Exception:
                pass

            V = mkt_cap + total_debt
            if V > 1e6:
                wacc = re * (mkt_cap/V) + rd * (1 - tax) * (total_debt/V)
                return float(np.clip(wacc, 0.04, 0.25))

        return float(re)

    def _estimate_tax_rate(self) -> float:
        inc = self._inc
        if not all(c in inc.columns for c in ["pretaxIncome","incomeTaxExpense"]):
            return 0.21
        df = inc[["pretaxIncome","incomeTaxExpense"]].apply(pd.to_numeric, errors="coerce")
        df = df[df["pretaxIncome"] > 0].dropna()
        if df.empty: return 0.21
        return float((df["incomeTaxExpense"] / df["pretaxIncome"]).clip(0, 0.40).median())

    def compute_eva_spread(self) -> dict:
        """
        EVA spread = ROIC_TTM - WACC
        ROIC = NOPAT_TTM / Invested_Capital  (same formula as DCFModel)
        Also computes ROE_Re_spread for moat classification.
        """
        wacc = self.compute_wacc_simple()
        tax  = self._estimate_tax_rate()
        inc  = self._inc.copy()
        bs   = self._bs.copy()

        if "operatingIncome" not in inc.columns:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        inc["nopat_q"] = pd.to_numeric(inc["operatingIncome"], errors="coerce") * (1 - tax)
        inc = inc.set_index("date").sort_index()

        for col in ["totalStockholdersEquity","totalDebt","cashAndCashEquivalents"]:
            bs[col] = pd.to_numeric(bs.get(col, 0), errors="coerce").fillna(0)
        bs["ic"] = bs["totalStockholdersEquity"] + bs["totalDebt"] - bs["cashAndCashEquivalents"]
        bs = bs.set_index("date")[["ic"]].sort_index()

        merged = bs.join(inc[["nopat_q"]], how="inner").dropna()
        if len(merged) < 4:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        eva_series = []
        dates = sorted(merged.index)
        for i, dt in enumerate(dates):
            if i < 3: continue
            nopat_ttm = float(merged["nopat_q"].iloc[i-3:i+1].sum())
            ic_snap   = float(merged["ic"].iloc[i])
            if ic_snap <= 0: continue
            roic_q  = nopat_ttm / ic_snap
            eva_q   = roic_q - wacc
            eva_series.append({"date": dt, "roic": roic_q, "eva_spread": eva_q, "ic": ic_snap})

        if not eva_series:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        latest    = eva_series[-1]
        recent_20 = eva_series[-20:]
        n_pos     = sum(1 for e in recent_20 if e["eva_spread"] > 0)

        # ROE - Re spread (for RIM moat classification)
        re = self._re or 0.10
        roe_re_spread = np.nan
        if "netIncome" in self._inc.columns and "totalStockholdersEquity" in self._bs.columns:
            inc2 = self._inc.copy()
            bs2  = self._bs.copy()
            inc2["ni"] = pd.to_numeric(inc2["netIncome"], errors="coerce")
            bs2["eq"]  = pd.to_numeric(bs2["totalStockholdersEquity"], errors="coerce")
            m = inc2[["date","ni"]].merge(bs2[["date","eq"]], on="date", how="inner").dropna()
            m = m[m["eq"] > 0]
            if len(m) >= 4:
                ni_ttm  = m["ni"].iloc[-4:].sum()
                eq_avg  = m["eq"].iloc[-4:].mean()
                roe_ttm = ni_ttm / eq_avg
                roe_re_spread = float(roe_ttm - re)

        if self.verbose:
            log(self.ticker,
                f"EVA: ROIC={latest['roic']:.2%}  WACC={wacc:.2%}  "
                f"spread={latest['eva_spread']:+.2%}  n_pos={n_pos}/20  "
                f"ROE-Re={roe_re_spread:+.2%}" if not np.isnan(roe_re_spread) else
                f"EVA: ROIC={latest['roic']:.2%}  WACC={wacc:.2%}  spread={latest['eva_spread']:+.2%}")

        result = {
            "roic":          latest["roic"],
            "wacc":          wacc,
            "eva_spread":    latest["eva_spread"],
            "roe_re_spread": roe_re_spread,
            "n_positive":    n_pos,
            "eva_series":    eva_series,
        }
        self._eva_cache = result
        return result

    def moat_to_omega_and_years(self, eva: dict) -> tuple:
        """
        EVA spread + ROE-Re spread → (omega, n_phase2_years, moat_label)

        Ohlson (1995): RI(t) = omega * RI(t-1)  →  omega ≡ rho (persistence)
        EVA spread classifies moat; same academic basis as DCFModel._moat_to_rho_and_years()

        Wide moat  : EVA>15% AND n_pos>=15  →  omega=0.90  Phase2=15yr
        Narrow moat: EVA 8-15% AND n_pos>=12 →  omega=0.83  Phase2=12yr
        Some moat  : EVA 3-8%  AND n_pos>=8  →  omega=0.75  Phase2= 8yr
        No moat    : EVA<3%    OR  n_pos<8   →  omega=0.60  Phase2= 5yr
        """
        if np.isnan(eva.get("eva_spread", np.nan)):
            return 0.75, 8, "Unknown (fallback)"

        spread = eva["eva_spread"]
        n_pos  = eva["n_positive"]

        if spread > 0.15 and n_pos >= 15:
            return 0.90, 15, "Wide moat"
        elif spread > 0.08 and n_pos >= 12:
            return 0.83, 12, "Narrow moat"
        elif spread > 0.03 and n_pos >= 8:
            return 0.75,  8, "Some moat"
        else:
            return 0.60,  5, "No moat"

    # ─────────────────────────────────────────────────────────
    # 4. RI Path Computation
    # ─────────────────────────────────────────────────────────

    def compute_ri_path(self) -> "RIMModel":
        """
        Full RI time-series:
          Phase 1: DuPont ROE forecast → RI = (ROE - Re) × BV
          Phase 2: AR(1) decay  RI(t) = omega × RI(t-1)
                   BV continues to grow: BV(t) = BV(t-1) × (1 + Re × retention)
                   (conservative: use Re as floor growth rate for BV in Phase 2)

        Academic basis:
          Phase 1: Sales-driven DuPont (Soliman 2008 TAR — DuPont predictability)
          Phase 2: Ohlson (1995) LIM  ω ∈ [0,1]
                   omega from EVA moat (Fama-French 1995 mean reversion)
        """
        re        = self._re
        coefs     = self.estimate_dupont_coefs()
        eva       = self.compute_eva_spread()
        omega, n_phase2, moat_label = self.moat_to_omega_and_years(eva)

        # ── Phase 1 ───────────────────────────────────────────
        ph1_df, bv_after_ph1 = self.forecast_roe_phase1(coefs)
        retention = self.estimate_retention()

        rows = ph1_df.to_dict("records")

        # ── Phase 2: AR(1) RI decay ──────────────────────────
        # Seed RI from last Phase 1 quarter
        ri_prev    = float(ph1_df["ri"].iloc[-1]) if not ph1_df.empty else 0.0
        bv_current = bv_after_ph1
        last_year  = int(ph1_df["year"].iloc[-1]) if not ph1_df.empty else datetime.now().year

        for t in range(1, n_phase2 + 1):
            # AR(1) decay on RI
            ri_t      = omega * ri_prev
            # RI spread = ri / bv (recover for record keeping)
            ri_spread = ri_t / bv_current if bv_current > 0 else 0.0
            roe_impl  = ri_spread + re     # implied ROE

            rows.append({
                "year":         last_year + t,
                "phase":        "ph2",
                "sales_annual": np.nan,
                "net_income":   np.nan,
                "npm":          np.nan,
                "at":           np.nan,
                "fl":           np.nan,
                "roe":          roe_impl,
                "re":           re,
                "ri_spread":    ri_spread,
                "bv_start":     bv_current,
                "ri":           ri_t,
            })
            ri_prev     = ri_t
            # BV grows conservatively: use Re as proxy growth for BV in Phase 2
            bv_current  = bv_current * (1 + max(roe_impl, 0) * retention)

        self.result_df = pd.DataFrame(rows)
        self._bv_terminal = bv_current
        self._ri_last     = ri_prev
        self._omega       = omega
        self._moat_label  = moat_label
        self._n_phase2    = n_phase2
        self._eva         = eva

        if self.verbose:
            log(self.ticker,
                f"RI path: Ph1={len(ph1_df)}yr Ph2={n_phase2}yr  "
                f"RI_last={ri_prev/1e6:.1f}M  [{moat_label} omega={omega}]")
        return self

    # ─────────────────────────────────────────────────────────
    # 5. Valuation
    # ─────────────────────────────────────────────────────────

    def compute_valuation(self) -> "RIMModel":
        """
        Ohlson (1995) RIM:
            P = BV_0 + PV(RI_Phase1) + PV(RI_Phase2) + PV(TV)
            TV = RI_last / (Re - g)   [if RI_last > 0 and Re > g]

        Discount rate: Re (cost of equity) — equity-side model.
        """
        re        = self._re
        g         = self.gdp_growth
        df        = self.result_df.copy()

        # ── Book Value at t=0 ─────────────────────────────────
        bs = self._bs.copy()
        bs["eq"] = pd.to_numeric(bs.get("totalStockholdersEquity", np.nan), errors="coerce")
        bv0 = float(bs.dropna(subset=["eq"])["eq"].iloc[-1])

        # ── PV of all RI rows ─────────────────────────────────
        pv_ri_total = 0.0
        pv_rows     = []
        for t, (_, row) in enumerate(df.iterrows(), 1):
            pv = row["ri"] / (1 + re) ** t
            pv_ri_total += pv
            pv_rows.append(pv)
        df["pv_ri"] = pv_rows

        T = len(df)

        # ── Terminal Value ────────────────────────────────────
        ri_last = self._ri_last
        tv      = 0.0
        if ri_last > 0 and re > g:
            tv = ri_last / (re - g)
        elif ri_last > 0:
            tv = ri_last * 10   # conservative multiple fallback
        pv_tv = tv / (1 + re) ** T if tv != 0 else 0.0

        # ── Intrinsic Value ───────────────────────────────────
        intrinsic = bv0 + pv_ri_total + pv_tv

        # ── Shares ────────────────────────────────────────────
        inc = self._inc
        shares = np.nan
        for col in ["weightedAverageShsOutDil", "weightedAverageShsOut"]:
            if col in inc.columns:
                s = pd.to_numeric(inc[col].iloc[-1], errors="coerce")
                if pd.notna(s) and s > 0:
                    shares = float(s); break

        target_price = intrinsic / shares if (not np.isnan(shares) and shares > 0) else np.nan

        # ── Current Price ─────────────────────────────────────
        current_price = np.nan
        try:
            r = requests.get(f"{FMP_BASE}/quote/{self.ticker}",
                             params={"apikey": self.api_key}, timeout=10)
            data = r.json()
            if isinstance(data, list) and data:
                current_price = float(data[0].get("price", np.nan) or np.nan)
        except Exception:
            pass

        upside = ((target_price / current_price) - 1) * 100 if (
            not np.isnan(target_price) and not np.isnan(current_price) and current_price > 0
        ) else np.nan

        tv_wt = pv_tv / (bv0 + pv_ri_total + pv_tv) * 100 if (bv0 + pv_ri_total + pv_tv) != 0 else np.nan

        self.result_df = df
        self.valuation = {
            "ticker":          self.ticker,
            "re":              re,
            "g_terminal":      g,
            "bv0":             bv0,
            "pv_ri":           pv_ri_total,
            "terminal_value":  tv,
            "pv_tv":           pv_tv,
            "tv_weight_pct":   tv_wt,
            "intrinsic_value": intrinsic,
            "shares":          shares,
            "target_price":    target_price,
            "current_price":   current_price,
            "upside_pct":      upside,
            "moat_label":      self._moat_label,
            "omega":           self._omega,
            "n_phase2":        self._n_phase2,
            "eva_spread":      self._eva.get("eva_spread", np.nan),
            "roic":            self._eva.get("roic",       np.nan),
            "roe_re_spread":   self._eva.get("roe_re_spread", np.nan),
            "eva_series":      self._eva.get("eva_series", []),
        }

        if self.verbose:
            log(self.ticker,
                f"RIM  BV={bv0/1e9:.2f}B  PV(RI)={pv_ri_total/1e9:.2f}B  "
                f"PV(TV)={pv_tv/1e9:.2f}B  IV={intrinsic/1e9:.2f}B  "
                f"TP=${target_price:.2f}  Upside={upside:.1f}%")
        return self

    # ─────────────────────────────────────────────────────────
    # 6. DB Save
    # ─────────────────────────────────────────────────────────

    def save_to_db(self, run_date: str = None) -> int:
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v  = self.valuation
        df = self.result_df.copy()

        rows = []
        for _, row in df.iterrows():
            rows.append({
                "date":           run_date,
                "ticker":         self.ticker,
                "year_label":     str(row.get("year", "")),
                "phase":          row.get("phase", ""),
                "sales_forecast": row.get("sales_annual") if pd.notna(row.get("sales_annual")) else None,
                "npm_forecast":   row.get("npm") if pd.notna(row.get("npm")) else None,
                "asset_turnover": row.get("at") if pd.notna(row.get("at")) else None,
                "fin_leverage":   row.get("fl") if pd.notna(row.get("fl")) else None,
                "roe_forecast":   row.get("roe") if pd.notna(row.get("roe")) else None,
                "re":             row.get("re"),
                "ri_spread":      row.get("ri_spread") if pd.notna(row.get("ri_spread")) else None,
                "bv_start":       row.get("bv_start") if pd.notna(row.get("bv_start")) else None,
                "ri":             row.get("ri") if pd.notna(row.get("ri")) else None,
                "pv_ri":          row.get("pv_ri") if pd.notna(row.get("pv_ri")) else None,
                "moat_label":     v["moat_label"],
                "rho":            v["omega"],
                "n_phase2":       v["n_phase2"],
                "g_terminal":     v["g_terminal"],
                "pv_all_ri":      v["pv_ri"],
                "terminal_value": v["terminal_value"],
                "intrinsic_value":v["intrinsic_value"],
                "current_price":  v["current_price"],
                "upside_pct":     v["upside_pct"],
                "target_price":   v["target_price"],
            })

        save_df = pd.DataFrame(rows).where(pd.notnull(pd.DataFrame(rows)), None)

        sql = f"""
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, year_label, phase,
             sales_forecast, npm_forecast, asset_turnover, fin_leverage,
             roe_forecast, re, ri_spread, bv_start, ri, pv_ri,
             moat_label, rho, n_phase2, g_terminal,
             pv_all_ri, terminal_value, intrinsic_value,
             current_price, upside_pct, target_price)
            VALUES
            (%(date)s, %(ticker)s, %(year_label)s, %(phase)s,
             %(sales_forecast)s, %(npm_forecast)s, %(asset_turnover)s, %(fin_leverage)s,
             %(roe_forecast)s, %(re)s, %(ri_spread)s, %(bv_start)s, %(ri)s, %(pv_ri)s,
             %(moat_label)s, %(rho)s, %(n_phase2)s, %(g_terminal)s,
             %(pv_all_ri)s, %(terminal_value)s, %(intrinsic_value)s,
             %(current_price)s, %(upside_pct)s, %(target_price)s)
            ON DUPLICATE KEY UPDATE
                ri=VALUES(ri), roe_forecast=VALUES(roe_forecast),
                target_price=VALUES(target_price), upside_pct=VALUES(upside_pct),
                intrinsic_value=VALUES(intrinsic_value), moat_label=VALUES(moat_label)
        """
        conn = get_conn()
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, save_df.to_dict("records"))
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────
    # 7. Visualisation
    # ─────────────────────────────────────────────────────────

    def plot(self) -> None:
        """
        2x2 chart:
          (0,0) Historical ROE vs Re + Phase1/2 forecast
          (0,1) RI time-series (historical + forecast)
          (1,0) EV composition: BV / PV(RI) / PV(TV)
          (1,1) Valuation summary text
        """
        if self.result_df is None or self.valuation is None:
            print("[WARN] Run compute_ri_path() and compute_valuation() first")
            return

        df = self.result_df.copy()
        v  = self.valuation
        re = v["re"]

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f"{self.ticker}  RIM Valuation  [{v['moat_label']}]",
                     fontsize=14, fontweight="bold")

        # ── (0,0) ROE vs Re ───────────────────────────────────
        ax = axes[0, 0]
        # Historical ROE (TTM)
        if self._inc is not None and "netIncome" in self._inc.columns and            self._bs is not None and "totalStockholdersEquity" in self._bs.columns:
            inc_h = self._inc.copy()
            bs_h  = self._bs.copy()
            inc_h["ni"] = pd.to_numeric(inc_h["netIncome"], errors="coerce")
            bs_h["eq"]  = pd.to_numeric(bs_h["totalStockholdersEquity"], errors="coerce")
            m = inc_h[["date","ni"]].merge(bs_h[["date","eq"]], on="date", how="inner").dropna()
            m = m[m["eq"] > 0].tail(20)
            if len(m) >= 4:
                roe_hist = []
                for i in range(3, len(m)):
                    ni_ttm = m["ni"].iloc[i-3:i+1].sum()
                    eq_avg = m["eq"].iloc[i-3:i+1].mean()
                    roe_hist.append({"date": m["date"].iloc[i], "roe": ni_ttm / eq_avg})
                roe_h_df = pd.DataFrame(roe_hist)
                ax.plot(roe_h_df["date"], roe_h_df["roe"] * 100,
                        color="#2980b9", lw=1.5, marker="o", ms=3, label="Historical ROE (TTM)")
                ax.axvline(roe_h_df["date"].iloc[-1], color="gray", lw=1, ls="--")

        ph1 = df[df["phase"] == "ph1"]
        ph2 = df[df["phase"] == "ph2"]
        ax.plot(range(1, len(ph1)+1), ph1["roe"]*100, "s-", color="#27ae60", lw=2,
                label=f"Phase1 ROE (DuPont)")
        ax.plot(range(len(ph1)+1, len(ph1)+len(ph2)+1), ph2["roe"]*100,
                "^--", color="#8e44ad", lw=2, label=f"Phase2 ROE (AR(1) implied)")
        ax.axhline(re * 100, color="#e74c3c", lw=1.5, ls=":",
                   label=f"Re={re*100:.1f}%")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title("ROE vs Cost of Equity (Re)")
        ax.set_ylabel("ROE / Re (%)")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # ── (0,1) RI time-series ──────────────────────────────
        ax = axes[0, 1]
        ph1_ri = ph1["ri"].values / 1e9
        ph2_ri = ph2["ri"].values / 1e9
        yr_ph1 = list(range(1, len(ph1)+1))
        yr_ph2 = list(range(len(ph1)+1, len(ph1)+len(ph2)+1))

        c_ph1 = ["#27ae60" if r >= 0 else "#e74c3c" for r in ph1_ri]
        c_ph2 = ["#8e44ad" if r >= 0 else "#c0392b" for r in ph2_ri]
        ax.bar(yr_ph1, ph1_ri, color=c_ph1, alpha=0.85, label="Phase1 RI", edgecolor="white")
        ax.bar(yr_ph2, ph2_ri, color=c_ph2, alpha=0.65, label="Phase2 RI", edgecolor="white")
        ax.axhline(0, color="black", lw=0.8)
        ax.axvline(len(ph1) + 0.5, color="gray", lw=1, ls=":")

        pv_tv_b = v["pv_tv"] / 1e9
        ax.axhline(pv_tv_b, color="#e67e22", lw=1.5, ls="--",
                   label=f"PV(TV)=${pv_tv_b:.1f}B")
        ax.set_title("Residual Income by Year (B$)")
        ax.set_ylabel("USD Billion")
        ax.set_xlabel("Forecast Year")
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ── (1,0) Value composition bar ───────────────────────
        ax = axes[1, 0]
        bv0    = v["bv0"]
        pv_ri  = v["pv_ri"]
        pv_tv  = v["pv_tv"]
        iv     = v["intrinsic_value"]

        components = {"Book Value": bv0/1e9, "PV(RI)": pv_ri/1e9, "PV(TV)": pv_tv/1e9}
        colors_bar  = {"Book Value": "#3498db", "PV(RI)": "#27ae60", "PV(TV)": "#e67e22"}
        bottom = 0.0
        for comp, val in components.items():
            if val > 0:
                ax.bar("Intrinsic Value", val, bottom=bottom, color=colors_bar[comp],
                       alpha=0.85, edgecolor="white", label=f"{comp} ${val:.1f}B")
                ax.text(0, bottom + val/2, f"{comp}\n${val:.1f}B",
                        ha="center", va="center", fontsize=9, color="white", fontweight="bold")
                bottom += val
        if not np.isnan(v["current_price"]) and not np.isnan(v["shares"]) and v["shares"] > 0:
            mkt_cap = v["current_price"] * v["shares"] / 1e9
            ax.axhline(mkt_cap, color="#e74c3c", lw=2, ls="--",
                       label=f"Market Cap ${mkt_cap:.1f}B")
        ax.set_title(f"Intrinsic Value Composition\n(IV=${iv/1e9:.1f}B)")
        ax.set_ylabel("USD Billion")
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ── (1,1) Valuation summary text ──────────────────────
        ax = axes[1, 1]
        ax.axis("off")
        tp   = v["target_price"]
        cp   = v["current_price"]
        up   = v["upside_pct"]
        evs  = v["eva_spread"]
        rrs  = v["roe_re_spread"]

        def _f(x, u="$", d=2):
            if x is None or (isinstance(x, float) and np.isnan(x)): return "N/A"
            if u == "$":  return f"${x:,.{d}f}"
            if u == "B":  return f"${x/1e9:.{d}f}B"
            if u == "%":  return f"{x*100:.{d}f}%"
            return str(x)

        lines = [
            f"Current Price  : {_f(cp)}",
            f"Target Price   : {_f(tp)}",
            f"Upside         : {up:.1f}%" if not np.isnan(up) else "Upside : N/A",
            "─" * 32,
            f"Book Value     : {_f(bv0,'B')}",
            f"PV(RI)         : {_f(pv_ri,'B')}",
            f"PV(TV)         : {_f(pv_tv,'B')}",
            f"Intrinsic Val  : {_f(iv,'B')}",
            "─" * 32,
            f"Re             : {_f(re,'%')}",
            f"g_terminal     : {_f(v['g_terminal'],'%')}",
            f"Moat           : {v['moat_label']}",
            f"omega (AR1)    : {v['omega']:.3f}",
            f"Phase2         : {v['n_phase2']} yr",
            "─" * 32,
            f"EVA spread     : {evs*100:+.1f}%" if not np.isnan(evs) else "EVA spread : N/A",
            f"ROE-Re spread  : {rrs*100:+.1f}%" if not np.isnan(rrs) else "ROE-Re : N/A",
        ]
        ax.text(0.05, 0.97, "\n".join(lines), transform=ax.transAxes,
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))
        plt.tight_layout()
        plt.show()

    # ─────────────────────────────────────────────────────────
    # 8. Convenience: run all
    # ─────────────────────────────────────────────────────────

    def run(self) -> "RIMModel":
        self.load_sales()
        self.load_financials()
        self.load_re()
        self.compute_ri_path()
        self.compute_valuation()
        return self


# ── Wrapper ───────────────────────────────────────────────────────────────────
def process_one_ticker_rim(
    ticker: str,
    engine,
    verbose: bool = False,
    run_date: str = None,
    save_db: bool = True,
) -> Dict[str, Any]:
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        model = RIMModel(ticker=ticker, engine=engine, verbose=verbose)
        model.run()
        rows = model.save_to_db(run_date) if save_db else 0
        v    = model.valuation
        return {
            "status":       "ok",
            "ticker":       ticker,
            "target_price": v.get("target_price", np.nan),
            "upside_pct":   v.get("upside_pct",   np.nan),
            "re":           v.get("re",            np.nan),
            "moat_label":   v.get("moat_label",    ""),
            "omega":        v.get("omega",          np.nan),
            "n_phase2":     v.get("n_phase2",       0),
            "eva_spread":   v.get("eva_spread",     np.nan),
            "rows_saved":   rows,
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": ticker,
            "target_price": np.nan, "upside_pct": np.nan,
            "re": np.nan, "moat_label": "", "omega": np.nan,
            "n_phase2": 0, "eva_spread": np.nan, "rows_saved": 0,
            "msg": str(e)[:150],
        }
    finally:
        gc.collect()


print("[OK] RIMModel class defined")

# ── Single-ticker test ────────────────────────────────────────────────────────
TEST_TICKER = "MSFT"
print(f"\n[Test] {TEST_TICKER} ...")
_res = process_one_ticker_rim(TEST_TICKER, engine, verbose=True)
print(f"  status={_res['status']}  TP={_res.get('target_price',float('nan')):.2f}  "
      f"Moat={_res.get('moat_label','')}  omega={_res.get('omega',float('nan')):.2f}")
if _res["status"] == "ok":
    _m = RIMModel(ticker=TEST_TICKER, engine=engine, verbose=True).run()
    _m.plot()


## Cell 4.5 · RIM Diagnostics

> Set `DIAG_TICKER`. Displays DuPont decomposition, RI path breakdown,
> EVA spread time-series, moat classification, and sensitivity heatmap.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cell 4.5 · RIM Diagnostics
# ═══════════════════════════════════════════════════════════════

DIAG_TICKER = "MSFT"   # <- change ticker

print(f"[Diagnostics] Running RIMModel for {DIAG_TICKER} ...")
_rm = RIMModel(ticker=DIAG_TICKER, engine=engine, verbose=True)
_rm.run()

v  = _rm.valuation
df = _rm.result_df.copy()

re        = v["re"]
bv0       = v["bv0"]
pv_ri     = v["pv_ri"]
pv_tv     = v["pv_tv"]
tv        = v["terminal_value"]
iv        = v["intrinsic_value"]
tp        = v["target_price"]
cp        = v["current_price"]
moat      = v["moat_label"]
omega     = v["omega"]
n_ph2     = v["n_phase2"]
eva_s     = v["eva_spread"]
roe_re    = v["roe_re_spread"]

SEP = "=" * 66
print(f"\n{SEP}")
print(f"  {DIAG_TICKER}  RIM Diagnostics")
print(SEP)

print(f"\n[1] Cost of Equity")
print(f"    Re (from DB)  : {re*100:.4f}%")
print(f"    g_terminal    : {GDP_GROWTH*100:.2f}%")
print(f"    Re - g        : {(re-GDP_GROWTH)*100:.4f}%")

print(f"\n[2] Phase 1 — DuPont ROE Forecast")
ph1 = df[df["phase"]=="ph1"]
for _, row in ph1.iterrows():
    ri_b = row["bv_start"]/1e9
    print(f"    Year {row['year']}  "
          f"Sales={row['sales_annual']/1e9:.2f}B  "
          f"NPM={row['npm']*100:.1f}%  AT={row['at']:.2f}  FL={row['fl']:.1f}  "
          f"ROE={row['roe']*100:.1f}%  "
          f"BV={ri_b:.2f}B  RI={row['ri']/1e9:.3f}B")

print(f"\n[3] Phase 2 — AR(1) RI Decay  (omega={omega:.3f}  [{moat}]  {n_ph2}yr)")
ph2 = df[df["phase"]=="ph2"]
for _, row in ph2.iterrows():
    print(f"    Year {row['year']}  "
          f"RI={row['ri']/1e9:.3f}B  "
          f"ROE_impl={row['roe']*100:.1f}%  "
          f"spread={row['ri_spread']*100:+.1f}%  "
          f"BV={row['bv_start']/1e9:.2f}B")

print(f"\n[4] Terminal Value")
tv_str = f"${tv/1e9:.3f}B" if tv > 0 else "0 (RI_last <= 0)"
print(f"    RI_last  : ${_rm._ri_last/1e9:.3f}B")
print(f"    TV = RI_last / (Re - g) = {tv_str}")
print(f"    PV(TV)   : ${pv_tv/1e9:.3f}B")

print(f"\n[5] Value Breakdown")
bv_wt = bv0 / iv * 100 if iv != 0 else float("nan")
ri_wt = pv_ri / iv * 100 if iv != 0 else float("nan")
tv_wt = pv_tv / iv * 100 if iv != 0 else float("nan")
print(f"    Book Value : ${bv0/1e9:.3f}B  ({bv_wt:.1f}%)")
print(f"    PV(RI)     : ${pv_ri/1e9:.3f}B  ({ri_wt:.1f}%)")
print(f"    PV(TV)     : ${pv_tv/1e9:.3f}B  ({tv_wt:.1f}%)")
print(f"    Intrinsic  : ${iv/1e9:.3f}B")

print(f"\n[6] Target Price")
tp_s  = f"${tp:.2f}" if not np.isnan(tp) else "N/A"
up_s  = f"{v['upside_pct']:.1f}%" if not np.isnan(v["upside_pct"]) else "N/A"
print(f"    TP={tp_s}  Current=${cp:.2f}  Upside={up_s}")

print(f"\n[7] EVA / Moat")
print(f"    EVA spread   : {eva_s*100:+.2f}%" if not np.isnan(eva_s) else "    EVA spread : N/A")
print(f"    ROE-Re spread: {roe_re*100:+.2f}%" if not np.isnan(roe_re) else "    ROE-Re : N/A")
print(f"    Moat grade   : {moat}")
print(f"    omega (AR1)  : {omega:.3f}")
print(f"    Phase2 period: {n_ph2} yr")
print(f"\n{SEP}\n")

# ── Visualisation ─────────────────────────────────────────────
_rm.plot()

# ── EVA time-series chart ─────────────────────────────────────
eva_series = v.get("eva_series", [])
if eva_series:
    wacc_val = _rm._eva.get("wacc", re)
    fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
    fig3.suptitle(f"{DIAG_TICKER}  EVA Spread & Moat  [{moat}]", fontsize=12, fontweight="bold")

    ax = axes3[0]
    dates_e = [e["date"] for e in eva_series]
    spreads = [e["eva_spread"]*100 for e in eva_series]
    roics   = [e["roic"]*100 for e in eva_series]
    ax.fill_between(dates_e, spreads, 0, where=[s>0 for s in spreads],
                    color="#2ecc71", alpha=0.25, label="EVA > 0")
    ax.fill_between(dates_e, spreads, 0, where=[s<=0 for s in spreads],
                    color="#e74c3c", alpha=0.25, label="EVA < 0")
    ax.plot(dates_e, spreads, color="#2980b9", lw=1.5, label="EVA spread (ROIC-WACC)")
    ax.plot(dates_e, roics, color="#8e44ad", lw=1.0, ls="--", alpha=0.7, label="ROIC TTM")
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(wacc_val*100, color="#e67e22", lw=1.2, ls=":", label=f"WACC={wacc_val*100:.1f}%")
    ax.set_title("EVA Spread Time-Series (ROIC - WACC)")
    ax.set_ylabel("%")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0f}%"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes3[1]
    # Sensitivity: Re x g → TP heatmap
    re_arr = np.linspace(max(re-0.03, 0.04), re+0.03, 7)
    g_arr  = np.linspace(max(GDP_GROWTH-0.015, 0.01), GDP_GROWTH+0.015, 5)
    tp_mat = np.full((len(g_arr), len(re_arr)), np.nan)
    all_ri = df["ri"].values
    T_tot  = len(all_ri)
    ri_last_val = _rm._ri_last
    shares_val  = v["shares"]

    for ri_i, re_ in enumerate(re_arr):
        for g_i, g_ in enumerate(g_arr):
            if re_ <= g_: continue
            pv_ri_ = sum(all_ri[t] / (1+re_)**(t+1) for t in range(T_tot))
            tv_    = ri_last_val / (re_ - g_) if ri_last_val > 0 else 0
            pv_tv_ = tv_ / (1+re_)**T_tot
            iv_    = bv0 + pv_ri_ + pv_tv_
            tp_mat[g_i, ri_i] = iv_ / shares_val if (not np.isnan(shares_val) and shares_val > 0) else np.nan

    _valid = tp_mat[~np.isnan(tp_mat)]
    if len(_valid) > 0:
        im = ax.imshow(tp_mat, aspect="auto", cmap="RdYlGn",
                       vmin=np.nanpercentile(tp_mat, 10),
                       vmax=np.nanpercentile(tp_mat, 90), origin="lower")
        plt.colorbar(im, ax=ax, label="Target Price ($)")
        for gi in range(len(g_arr)):
            for ri_i in range(len(re_arr)):
                val = tp_mat[gi, ri_i]
                if not np.isnan(val):
                    ax.text(ri_i, gi, f"${val:.0f}", ha="center", va="center", fontsize=7)
        ax.set_xticks(range(len(re_arr)))
        ax.set_xticklabels([f"{r*100:.1f}%" for r in re_arr], rotation=45, fontsize=7)
        ax.set_yticks(range(len(g_arr)))
        ax.set_yticklabels([f"{g_*100:.1f}%" for g_ in g_arr], fontsize=7)
        ci_b = np.argmin(np.abs(re_arr - re))
        gi_b = np.argmin(np.abs(g_arr  - GDP_GROWTH))
        ax.plot(ci_b, gi_b, marker="*", ms=14, color="white",
                markeredgecolor="black", markeredgewidth=0.8,
                label=f"Base TP={f'${tp:.0f}' if not np.isnan(tp) else 'N/A'}")
        ax.legend(fontsize=8)
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "All TP values invalid", transform=ax.transAxes,
                ha="center", va="center", fontsize=11)
    ax.set_xlabel("Re (cost of equity)")
    ax.set_ylabel("g_terminal")
    ax.set_title("TP Sensitivity (Re x g_terminal)")

    plt.tight_layout()
    plt.show()


## Cell 5 · Batch Run

> Adjust `TICKER_START / TICKER_END`, set `SKIP_DONE=True` to resume.


In [ ]:
RUN_TICKERS = US_TICKER_LIST[TICKER_START:TICKER_END]
total    = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH) as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = 0
results = []
t0 = time.time()

log("BATCH", f"Start: {total:,} tickers  run_date={run_date}  SKIP_DONE={SKIP_DONE}")
print("=" * 70)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct    = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP", flush=True); skip_cnt += 1; continue

    res = process_one_ticker_rim(ticker, engine, verbose=False,
                                 run_date=run_date, save_db=True)
    results.append(res)

    if res["status"] == "ok":
        tp  = res["target_price"]
        up  = res["upside_pct"]
        tp_s = f"TP=${tp:.2f}" if not np.isnan(tp) else "TP=N/A"
        up_s = f"Upside={up:.1f}%" if not np.isnan(up) else ""
        print(f"{prefix} OK  {tp_s} {up_s}  Re={res['re']:.3f}  "
              f"Moat={res['moat_label']}  omega={res['omega']:.2f}", flush=True)
        with open(DONE_PATH, "a") as f: f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res.get('msg','')}", flush=True)
        with open(FAIL_PATH, "a") as f: f.write(ticker + "\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 70)
log("BATCH", f"Done  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"elapsed={elapsed:.0f}s  avg={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

if results:
    summary = pd.DataFrame(results)
    ok_df   = summary[summary["status"]=="ok"].sort_values("upside_pct", ascending=False)
    print("\n[Top 20 Upside]")
    display(ok_df[["ticker","target_price","upside_pct","re","moat_label","omega","n_phase2"]].head(20))
    print("\n[Moat Distribution]")
    display(ok_df["moat_label"].value_counts())


## Cell 6 · Results & Visualisation

> `VIZ_TICKERS` to individual plot.


In [ ]:
# ── DB summary ────────────────────────────────────────────────────────────────
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT DATE(date) AS run_date,
                   COUNT(DISTINCT ticker) AS tickers,
                   COUNT(*) AS total_rows,
                   AVG(target_price) AS avg_tp,
                   AVG(upside_pct) AS avg_upside
            FROM {TABLE_RESULT}
            GROUP BY DATE(date)
            ORDER BY run_date DESC
            LIMIT 10
        """)
        _summary = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 60)
print(f"[DB Summary] {TABLE_RESULT}")
print("=" * 60)
display(_summary)

# ── Latest run upside distribution ────────────────────────────────────────────
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(f"SELECT MAX(date) AS max_date FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["max_date"]
        if max_date:
            cur.execute(f"""
                SELECT ticker,
                       MAX(target_price)  AS target_price,
                       MAX(current_price) AS current_price,
                       MAX(upside_pct)    AS upside_pct,
                       MAX(re)            AS re,
                       MAX(moat_label)    AS moat_label,
                       MAX(rho)           AS omega
                FROM {TABLE_RESULT}
                WHERE date=%s AND target_price IS NOT NULL
                GROUP BY ticker
                ORDER BY upside_pct DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

if max_date and not results_df.empty:
    print(f"\n[Latest run: {max_date}]  {len(results_df)} tickers")
    print("\nTop 20 Upside")
    display(results_df.head(20)[["ticker","target_price","current_price","upside_pct","re","moat_label","omega"]])
    print("\nBottom 20 Upside")
    display(results_df.tail(20)[["ticker","target_price","current_price","upside_pct","re","moat_label","omega"]])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"RIM Valuation Summary ({max_date})", fontsize=13)

    ax = axes[0]
    upside_clean = results_df["upside_pct"].dropna().clip(-200, 200)
    ax.hist(upside_clean, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0,  color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20% (Buy zone)")
    ax.set_title("Upside Distribution"); ax.set_xlabel("Upside (%)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    moat_counts = results_df["moat_label"].value_counts()
    moat_colors = {"Wide moat":"#27ae60","Narrow moat":"#2980b9",
                   "Some moat":"#f39c12","No moat":"#e74c3c","Unknown (fallback)":"#95a5a6"}
    colors_bar2 = [moat_colors.get(m, "#888") for m in moat_counts.index]
    moat_counts.plot(kind="barh", ax=ax, color=colors_bar2)
    ax.set_title("Moat Grade Distribution"); ax.grid(axis="x", alpha=0.3)

    ax = axes[2]
    for label, color in moat_colors.items():
        sub = results_df[results_df["moat_label"]==label]["upside_pct"].dropna().clip(-200,200)
        if not sub.empty:
            ax.hist(sub, bins=20, alpha=0.55, color=color, label=label, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title("Upside by Moat Grade"); ax.set_xlabel("Upside (%)"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

    plt.tight_layout(); plt.show()

# ── Individual plots ──────────────────────────────────────────────────────────
VIZ_TICKERS = ["AAPL", "MSFT", "NVDA"]   # <- change

print(f"\n[Individual Plot] {VIZ_TICKERS}")
for tk in VIZ_TICKERS:
    try:
        m = RIMModel(ticker=tk, engine=engine, verbose=False).run()
        m.plot()
    except Exception as e:
        print(f"  [{tk}] failed: {e}")
